In [ ]:
%load_ext autoreload
%autoreload 2

# Gaussian Scenes and Utilities

In [ ]:
#|default_exp nbx.gaussian_splatting
#|export
from typing import TypeAlias

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from jax.scipy.spatial.transform import Rotation as Rot

from psilia import get_console
from psilia.transforms import Transform
from psilia.vision import CameraIntrinsics, screen_from_camera

console = get_console()

In [ ]:
from psilia.plotting import RerunLogger
from psilia.timer import Timer
from psilia.vision.camera import render_naively as _render
from psilia.vision.camera import unproject as _unproject

timer = Timer()
rrl = RerunLogger("Splatting")

key = jax.random.PRNGKey(0)

intr = CameraIntrinsics.load("./_intr.yaml").resize(0.25)
W = intr.w
H = intr.h
console.print(intr)


project = jax.jit(lambda x, cam,: screen_from_camera(cam.inv()(x), intr))
unproject = jax.jit(lambda x: _unproject(x, intr))
render = jax.jit(
    lambda xs, vs, down=1: _render(xs, vs, intr, (intr.h, intr.w), down, jnp.array(1.)),
    static_argnums=(2,))
compute_depths = jax.jit(lambda xs, cam: cam.inv()(xs)[:,2])

## Representations and Sampling

In [ ]:
#|export
from jax.tree_util import register_pytree_node_class
from dataclasses import dataclass


@register_pytree_node_class
@dataclass
class Gaussian:
    def __init__(self, mean, covariance, color):
        self.mean = mean
        self.covariance = covariance
        self.color = color
        

    # Jax Pytree Registration
    def tree_flatten(self):
        return (
            (self.mean, self.covariance, self.color),
            {},
        )

    # Jax Pytree Registration
    @classmethod
    def tree_unflatten(cls, aux_data, children):
        return cls(*children, **aux_data)

    def __iter__(self):
        return iter(self.tree_flatten()[0])

    @property
    def mu(self):
        return self.mean

    @property
    def cov(self):
        return self.covariance

    @property
    def c(self):
        return self.color


@register_pytree_node_class
class GaussianScene:
    #TODO: implement __matmul__(self, tf:Transform)??
    def __init__(self, 
        means:jax.Array, 
        covariances:jax.Array, 
        colors:jax.Array, 
        weights:jax.Array
    ):
        self.means = means
        self.covariances = covariances
        self.colors = colors
        self.weights = weights

    def __getitem__(self, i):
        return Gaussian(
            self.means[i],
            self.covariances[i],
            self.colors[i],
            self.weights[i],
        )

    # Jax Pytree Registration
    def tree_flatten(self):
        return (
            (self.means, self.covariances, self.colors, self.weights),
            {},
        )

    # Jax Pytree Registration
    @classmethod
    def tree_unflatten(cls, aux_data, children):
        return cls(*children, **aux_data)

    @property
    def mu(self):
        return self.means

    @property
    def cov(self):
        return self.covariances

    @property
    def c(self):
        return self.colors

    @property
    def w(self):
        return self.weights

    def __iter__(self):
        return iter(self.tree_flatten()[0])


def get_scales_and_rot(cov):
    """Scale-vector and rotation matrix from covariance matrix."""
    sigma, U = jnp.linalg.eigh(cov)
    return jnp.sqrt(sigma), U


def sample_scales_and_quaternions(key, N=1, smin=0.1, smax=0.5):
    """Sample scale-vectors and quaternions."""
    _, key = jax.random.split(key)
    ss = jax.random.uniform(key, (N,3), minval=smin, maxval=smax)
    qs = jax.random.normal(key, (N,3))
    qs = qs / jnp.linalg.norm(qs, axis=-1, keepdims=True)
    return ss, qs

def sample_covariance(key, N=1, smin=0.1, smax=0.5):
    """Sample covariance matrices."""
    ss, qs = sample_scales_and_quaternions(key, N, smin, smax)
    rots = Rot.from_quat(qs).as_matrix()
    return jax.vmap(lambda s, rot: rot @ jnp.diag(s**2) @ rot.T)(ss, rots)

# TODO: Implement version of this taking a GaussianScene as input.
def sample_from_scene(key, mus, covs, cols, N=100, flat=True):
    """Sample points from a Gaussian scene."""
    _,key = jax.random.split(key)
    d = mus.shape[-1]
    xs = jax.random.multivariate_normal(key, mus, covs, (N, mus.shape[0]))
    xs = xs.transpose(1,0,2)
    cs = jnp.repeat(cols[:,None], N, axis=1)
    if flat:
        xs = xs.reshape(-1,d)
        cs = cs.reshape(-1,3)
    return xs, cs


In [ ]:
# Camera pointing along the x-axis
cam = Transform.from_te(
    jnp.array([-10, 0., 0.]),
    [-90, 0, -90], seq="xyz", degrees=True
)

# Sample a scene of N Gaussians (means, covariances, colors)
N = 10
key, key0, key1, key2 = jax.random.split(key, 4)
mus = jax.random.uniform(key0, (N,3), minval=-5, maxval=5)
# covs = sample_covariance(key1, N, smin=jnp.array([0.1, 0.1, 0.05]), smax=jnp.array([0.5, 2., 1.]))
covs = sample_covariance(key1, N, smin=jnp.array([0.1, 0.1, 0.1]), smax=jnp.array([0.5, .5, .5]))
cols = jax.random.uniform(key2, (N,3), minval=0., maxval=1.)

console.inspect(N=N, mus=mus, covs=covs, cols=cols)

key,_ = jax.random.split(key)
xs, cs = sample_from_scene(key, mus, covs, cols, 1_000)

timer()
im = jax.block_until_ready(
    render(cam.inv()(xs), cs, down=1))
timer("Naive Render", msg=xs.shape)
# =====================
plt.imshow(im, interpolation="none")

In [ ]:
# ==========================
rrl.set_time(0)
rrl.log_camera("camera", cam, intr)
rrl.log_points("points", xs, cs, radii=0.02)

## 2D Projections and Bounding boxes

In [ ]:
#|export
def project_gaussian(mu, cov, cam, intr):
    """
    Project a 3D Gaussian in world coordinates to the image plane.

    Args:
        mu: jnp.ndarray, shape (3,)
        cov: jnp.ndarray, shape (3,3)
        cam: psilia.transforms.Transform
        intr: psilia.vision.CameraIntrinsics
    Returns:
        mu2d: jnp.ndarray, shape (2,)
        cov2d: jnp.ndarray, shape (2,2)
        valid: bool
    """
    R = cam.rot.as_matrix()
    mu_ = cam.inv()(mu)
    mu2d, valid = screen_from_camera(mu_, intr)
    J = jax.jacobian(lambda x: screen_from_camera(x, intr)[0])(mu_)
    cov2d = J @ R.T @ cov @ R @ J.T
    return (mu2d, cov2d), valid


def bounding_box_from_gaussian(mu, cov, k=3):
    bb = jnp.array([
        mu - k*jnp.sqrt(jnp.diag(cov)),
        mu + k*jnp.sqrt(jnp.diag(cov))])
    return bb

In [ ]:
#|export
# JITed versions of scene-related functions
project_gausians = jax.jit(jax.vmap(project_gaussian, (0,0,None,None)), static_argnums=(3,))
bounding_boxes = jax.jit(jax.vmap(bounding_box_from_gaussian, (0,0)), static_argnums=(2,))

In [ ]:
timer()
(mus2d, covs2d), valid = jax.block_until_ready(
    project_gausians(mus, covs, cam, intr))
timer("Project Gaussians")

bbs = bounding_boxes(mus2d, covs2d).astype(jnp.int32)
key,_ = jax.random.split(key)
xs2d, cs2d = sample_from_scene(key, mus2d, covs2d, cols, 100)

console.inspect(xs2d=xs2d, cs2d=cs2d)

# =====================================
fig, axs = plt.subplots(1,2, figsize=(6,2))

axs[0].scatter(*project(xs, cam)[0].T, c=cs, s=1.)
axs[0].set_aspect("equal")
axs[0].set_xlim(0, intr.w)
axs[0].set_ylim(0, intr.h)
axs[0].invert_yaxis()

axs[1].set_aspect("equal")
axs[1].set_xlim(0, intr.w)
axs[1].set_ylim(0, intr.h)
axs[1].invert_yaxis()
axs[1].scatter(xs2d[:,0], xs2d[:,1], c=cs2d, s=1.)
for bb, c in zip(bbs, cols):
    axs[1].plot(
        [bb[0,0], bb[1,0], bb[1,0], bb[0,0], bb[0,0]],
        [bb[0,1], bb[0,1], bb[1,1], bb[1,1], bb[0,1]],
        linewidth=1,
        color=np.array(c)
    )

fig.tight_layout()

In [ ]:
#|export
# TODO: test and visualize this function
def quantize_and_clip_bbs(bbs, W, H):
    """Quantize bounding boxes to integer pixel coordinates.

    A quantized bounding box `b'` covers the pixels in a xy-index grid
    `grid[b'[0,0]:b'[1,0], b'[0,1]:b'[1,1]]`.
    """
    bbs = bbs.at[:,1].add(1.)
    bbs = jnp.floor(bbs).astype(jnp.int32)
    bbs = jnp.clip(bbs, 0, jnp.array([W, H]))
    return bbs

In [ ]:
key,_ = jax.random.split(key)
ll = jax.random.uniform(key, (5,2), minval=-1., maxval=2.)
ur = jax.random.uniform(key, (5,2), minval=2.1, maxval=5.1)

bbs = jnp.stack([ll, ur], axis=1)

bbs_ = quantize_and_clip_bbs(bbs, *intr.resolution)

fig, axs = plt.subplots(1, bbs.shape[0], figsize=(10,2))

for ax,bb,bb_ in zip(axs, bbs, bbs_):
    ax.set_aspect("equal")
    ax.set_xlim(-2, 12)
    ax.set_ylim(-2, 12)
    im = (jnp.arange(10)%2)[:,None]+(jnp.arange(10)%2)[None]
    im = 0.1*(im%2)

    im = im.at[bb_[0,1]:bb_[1,1], bb_[0,0]:bb_[1,0]].add(1)

    console.print("bb:", bb[1], "->", bb_[1])
    ax.imshow(im, extent=(0,10, 10,0), interpolation="none")
    ax.scatter(*bb.T, color="red", s=5)
    ax.plot(
        [bb[0,0], bb[1,0], bb[1,0], bb[0,0], bb[0,0]],
        [bb[0,1], bb[0,1], bb[1,1], bb[1,1], bb[0,1]],
        linewidth=1,
        color="magenta"
    )


### Ray projection

In [ ]:
Array: TypeAlias = jax.Array
Matrix: TypeAlias = Array
CovarianceMatrix: TypeAlias = Matrix
PrecisionMatrix: TypeAlias = Matrix
RayOrigin: TypeAlias = Array
RayDirection: TypeAlias = Array
Float: TypeAlias = Array


def bilinear(x: Array, y: Array, A: Matrix) -> Float:
    return x.T @ A @ y


def gaussian_restriction_to_ray(
    mu_tilde, prec: PrecisionMatrix, o: RayOrigin, v: RayDirection
):
    """
    Restricts a normalized Gaussian to a ray and returns
    the mean `mu` and standard deviation `sig`, such that,
    parameterizing they ray by $r(t) = o +t*v$, we have
    $$
        N( r(t) | \tilde\\mu, cov) = w * N(t | \\mu, \\sigma)
    $$
    where
    $$
        w = N( r(\\mu) | \tilde\\mu, cov) /  N(\\mu, \\mu, \\sigma).
    $$
    Note that $\\mu$ is the maximum of both the nominator and denominator.
    Also note that the first equation implies that the integral of
    the Gaussian along the ray is given by $w$.
    """
    mu1D = bilinear(mu_tilde - o, v, prec) / bilinear(v, v, prec)
    sig1D = 1 / jnp.sqrt(bilinear(v, v, prec))
    return mu1D, sig1D

# Rasterizer: Rasterize Bounding Boxes

In [ ]:
#|export
from functools import partial

# TODO: Add to documentation what happens if M is bigger or smaller than the sum of areas.
@partial(jax.jit, static_argnames=("M",))
def memory_layout(areas, M):
    """
    Computes the memory layout for a set of gaussians with given areas, that is 
    the starting memory slot for each gaussian, the gaussian index for each memory slot,
    and the relative offset within each gaussian's memory range.

    The range in memory allocated for gaussian `i` with area `a[i]` is
    $$
        range(m[i], m[i] + a[i]) = [m[i]+0, ..., m[i]+a[i]-1]
    $$
    If `j` is an index in memory, then `i = g[j]` is the index of
    the associated gaussian, and `k = off[j]` is the relative
    offset in the memory range of the gaussian.

    Example:
    Let's assume we have 3 gaussians with areas [3,2,2]
    The total memory size is M = 7. Then the memory and 
    gaussian index lookups are
    $$
        areas = [3,       2,    2   ]
        ind   = [0, 0, 0, 1, 1, 2, 2],
        off   = [0, 1, 2, 0, 1, 0, 1]
        mem   = [0,       3,    5   ],
    $$ 

    Args:
        areas (int[N]): Allocated size of memory for each gaussian
        M (int): Maximum memory size (make sure it is at least the sum of areas)
        
    Returns:
        mem (int[N]): Starting memory slot for each gaussian
        ind (int[M]): Gaussian index for each memory slot
        off (int[M]): Relative offset within each gaussian's memory range
    """
    mem = jnp.cumsum(areas) - areas
    ind = jnp.full(M, 0, dtype=jnp.int32).at[mem + areas].set(1)
    ind = jnp.cumsum(ind)
    off = jnp.cumsum(
        jnp.full(M, 1, dtype=jnp.int32).at[mem + areas].add(-areas)
    )
    off -= 1
    return mem, ind, off

In [ ]:
# Example 1:
# The size of the allocated memory is
# just as large as needed, i.e. the sum of areas.
areas = jnp.array([3,2,2])
M = areas.sum().item()
m, g, off = memory_layout(areas, M)
used = (jnp.arange(M) < m[-1] + areas[-1]).astype(jnp.int32)
console.print(f"""
[bold black]EXAMPLE 2:[/bold black] areas={areas}, M={M}
m    = {m}
used = {used}
g    = {g}
off  = {off}
""")

# Example 2:
# The size of the allocated memory is larger than needed. This is especially
# relevant for JAX to enable static shapes.
areas = jnp.array([3,2,2])
M = areas.sum().item() + 5
m, g, off = memory_layout(areas, M)
used = (jnp.arange(M) < m[-1] + areas[-1]).astype(jnp.int32)
console.print(f"""
[bold black]EXAMPLE 2:[/bold black] areas={areas}, M={M}
m    = {m}
used = {used}
g    = {g}
off  = {off}
""")

In [ ]:
#|export
from jax.tree_util import register_pytree_node_class
# TODO: Make this nice.
# TODO: We should change the interface for RaserizeResult, 
#   or even rename it. The Rasterizer computes a GaussianRegistry, i.e.
#   we compute for each pixel which Gaussians influence the pixel.
#   The input to the Rasterizer is basically a PixelRegistry, 
#   containing which pixels are covered by a Gaussian.
#       RaserizeResult.i(uv) -> MaskedGaussianIndices 
#       RaserizeResult.uv(i) -> MaskedPixels
# TODO: Add documentation
@register_pytree_node_class
class RasterizeResult:
    def __init__(self, counts2d, hooks2d, pxs, inds, order):
        self._counts2d = counts2d
        self._hooks2d = hooks2d
        self._counts = counts2d.reshape(-1)
        self._hooks = hooks2d.reshape(-1)
        self._pxs = pxs
        self._inds = inds
        self._order = order

    # Jax Pytree Registration
    def tree_flatten(self):
        return (
            (self._counts2d, self._hooks2d, self._pxs, self._inds, self._order),
            {},
        )

    # Jax Pytree Registration
    @classmethod
    def tree_unflatten(cls, aux_data, children):
        return cls(*children, **aux_data)

    def _i(self, uv_flat, max_raster_depth):
        valid = jnp.arange(max_raster_depth) < self._counts[uv_flat]
        sub = jax.lax.dynamic_slice(
            self._order, 
            jnp.array([self._hooks[uv_flat]]), (max_raster_depth,))
            
        return self._inds[sub], valid

    def i(self, uv, max_raster_depth):
        # uv_flat = jnp.ravel_multi_index(uv.astype(jnp.int32), self._shape, mode='clip')
        uv = uv.astype(jnp.int32)
        c = self._counts2d[uv[0], uv[1]]
        h = self._hooks2d[uv[0], uv[1]]
        valid = jnp.arange(max_raster_depth) < c
        sub = jax.lax.dynamic_slice(
            self._order, 
            jnp.array([h]), (max_raster_depth,))
            
        return self._inds[sub], valid

    def uv(self, i):
        uvs_flat = self._pxs[self._inds == i]
        uvs = jax.vmap(jnp.unravel_index)(uvs_flat, self._shape)
        return uvs, jnp.full(uvs.shape[:-1], True)

    def __len__(self):
        return len(self._inds)

    def __gt__(self, c):
        return self._counts > c

    def __eq__(self, c):
        return self._counts == c


# TODO: Rasterizer should take as input a PixelRegistry 
#   (an example are the bounding boxes for a Gaussian projection), 
#   and output a GaussianRegistry.
@partial(jax.jit, static_argnames=("M", "W", "H"))
def rasterize(bbs, zs, M, W, H):
    """
    Rasterize the bounding boxes of the gaussians.

    Args:
        bbs: jnp.ndarray, shape (N,2,2)
            The quantized bounding boxes of the Gaussians. 
            Each bounding box is specified for one Gaussian 
            as an array of its low-left and top-right coordinates.
        zs: jnp.ndarray, shape (N,)
            Array of depth values for each Gaussian, 
            used for sorting in depth order.
        M: int
            The size of the allocated memory buffer to 
            store Gaussian-to-pixel associations. 
            Typically at least the sum of the areas of 
            all bounding boxes, but may be larger for static shapes.
        W: int
            The width of the raster image (number of columns).
        H: int
            The height of the raster image (number of rows).
    Returns:
        counts: jnp.ndarray, shape (W*H,)
        hooks: jnp.ndarray, shape (W*H,)
        pixel_indices: jnp.ndarray, shape (M,)
        gaussian_inds : jnp.ndarray, shape (M,)
        order: jnp.ndarray, shape (M,)
    """
    # TODO: what if zs are negative, and
    #   uvs are outside the image bounds?

    bbs = quantize_and_clip_bbs(bbs, W, H)
    whs = bbs[:,1]-bbs[:,0]
    areas = whs[:,0]*whs[:,1]
    _, inds, off = memory_layout(areas, M)

    # Register Gaussians at pixels: 
    # That means register each pixel in each area in memory (int[M])
    # 
    # Turn the offsets into pixel indices,
    # combining the projected gaussians and the offsets.
    # For each gaussian take the influenced pixels and concat all
    # af them. This means some pixels occur multiple times.
    pixel_offsets_2D = jnp.stack(jax.vmap(jnp.unravel_index)(off, whs[inds]), axis=-1)
    pixel_indices_2D = bbs[inds,0] + pixel_offsets_2D
    pixel_indices = jnp.ravel_multi_index(pixel_indices_2D.T, (W,H), mode="clip")

    # Sort the pixels by pixel index and depth.
    # The ordered versions will look like
    # ```
    #   [  0,   0,   0,   1,   1, ...] <- pixel indices (after lexsort)
    #   [1.1, 1.5, 2.1, 0.9, 4.0, ...] <- depth values (after lexsort)
    # ```
    # TODO: This is the main runtime bottleneck.
    order = jnp.lexsort([zs[inds], pixel_indices])

    # Counts of Gaussians affecting each pixel in the images
    # and the hook from the pixel into the **ordered** memory
    counts = jnp.full(W*H, 0, dtype=jnp.int32)
    counts = counts.at[pixel_indices].add(1)
    hooks = jnp.full(W*H, -1, dtype=jnp.int32)
    hooks = hooks.at[jnp.arange(W*H)].set(jnp.cumsum(counts) - counts)

    return RasterizeResult(
        counts.reshape(W,H), hooks.reshape(W,H), pixel_indices, inds, order)

In [ ]:
bbs = bounding_boxes(mus2d, covs2d)
bbs_ = quantize_and_clip_bbs(bbs, intr.w, intr.h)
whs = bbs_[:,1]-bbs_[:,0]
M = jnp.sum(whs[:,0]*whs[:,1]).item()

zs = compute_depths(mus, cam)
rasterize_result = rasterize(bbs, zs, M, intr.w, intr.h)

u_inds = np.where(rasterize_result > 1)[0]

In [ ]:
# TODO: this should go to vision.camera, 
#   or even as a method of CameraIntrinsics
def compute_uv_grid(intr, flat=False, indexing="uv"):
    if indexing == "uv" or indexing=="xy":
        u, v = jnp.mgrid[:intr.w, :intr.h]
    elif indexing == "ij":
        u, v = jnp.mgrid[:intr.h, :intr.w]
    else:
        raise ValueError(f"Unknown indexing: {indexing}")
    
    uv_grid = jnp.stack([u, v], axis=-1) + 0.5
    if flat:
        uv_grid = uv_grid.reshape(-1, 2)
    return uv_grid

In [ ]:
uv_grid = compute_uv_grid(intr, indexing="uv")
uv_grid.shape, (W,H)

# Renderering Model

### Gaussian splatting Equation
See [Beer-Lambert_law](https://en.wikipedia.org/wiki/Beer%E2%80%93Lambert_law):
- "Optical density": $\sigma \in [0,\infty]$
- "Opacity": $\alpha = 1 - \exp(-\sigma) \in [0,1]$
- "Transmittance": $\exp(-\sigma) = (1 - \alpha) \in [0,1]$

The Gaussian splatting equation is an discrete apprimation of the volumetric rendering equation.
For a pixel $u$ we render the color $c(u)$ as folows (...)
$$
    c(u) = \sum_i T_i \cdot \alpha_i \cdot c_i.
$$
Here we have
$$
\begin{align*}
T_i &= \prod_{j<i} (1-\alpha_j)& \\
\alpha_i &= 1 - \exp(-\sigma_i)& \\
\sigma_i &= w_i \cdot \exp\big(-\tfrac{1}{2} (u - \mu_i)^T\Sigma_i^{-1}(u - \mu_i) \big)&
\end{align*}
$$

### Probabilistic Splatting Model

Let $c$ be a color (or more generally an observed feature) and $u$ a pixel (or more generally a point on the sensor canvas). Let $\textcolor{red}{I = I(u)}$ be the set of Gaussian indices registered for pixel $u$; **For ease of notation we assume that that the Gaussian indices are pre-ordered by the depth of the associated Gaussians.** 

**There are 2 ways of writing down the model:**

1. Specifying the "optical density" $\sigma(u ; i) \in [0,\infty]$ (in poisson process terms this means specifying the "rate") and then defining $\alpha = 1 - \exp(-\sigma) \in [0,1]$. Note that $\sigma$ is NOT normalized in general.

2. Or by directly defining the "opacity" $\alpha(u;i) \in [0,1]$, 
and inferring the "Optical density" (poisson rate) $\sigma$. 

Using scaled Gaussians to define the optical density induces a nice enough model class for the opacity; see the cell below.

We define
$$
\begin{align*}
    P(c, u) 
    &
         = \sum_I P(c,i,u) \ \textcolor{lightgray}{\underbrace{+ \ P(c,\infty,u)}_{\text{"Background"(omitted)}}}
    & \\
    &
        = P(u) \cdot \sum_I \textcolor{magenta}{P(c \mid  i, u)} \cdot \textcolor{blue}{P(i \mid u)}
    & \\
    &
        = 
        P(u) \cdot \sum_I \textcolor{magenta}{P(c \mid i, u)}
            \cdot \textcolor{blue}{
        \alpha(i,u) \cdot \prod_{j<i}\big(1-\alpha(i,u)\big)}
    & \\
    &
        = 
        P(u) \cdot \sum_I \textcolor{magenta}{c_i}
            \cdot \textcolor{blue}{
        \alpha_i \cdot T_i}
    &
\end{align*}
$$

### Notes

- Mixture -vs.- Poisson Process. 

In [ ]:
# Let's understand scaled Gaussian optical density rates
xs = jnp.linspace(-1., 1., 10_000)

def test_alphas(w, xs):
    sigmas = w*jnp.exp(-0.5*xs**2/0.1**2)
    alphas = 1 - jnp.exp(-sigmas)
    return alphas


fig, ax = plt.subplots(1,1, figsize=(8,2))
ax.set_title("Opacity of scaled Gaussians optical densities.")
ax.plot(xs, test_alphas(.1, xs), label="w=0.1", linewidth=0.5, color="C0")
ax.plot(xs, test_alphas(1., xs), label="w=1", linewidth=1, color="C0")
ax.plot(xs, test_alphas(10., xs), label="w=10", linewidth=1.5, color="C0")
ax.plot(xs, test_alphas(100., xs), label="w=100", linewidth=2, color="C0")
ax.legend();

#### Sampling

In [ ]:
#|export
from jax.scipy.stats.multivariate_normal import logpdf as mvnormal_logpdf, pdf as mvnormal_pdf
from jax.scipy.stats.norm import logpdf as normal_logpdf
from functools import partial


MAX_RASTER_DEPTH = 100

def _sample_pixel(key, uv_idx, uv_grid_flat, scene2d, zs, rasterize_result):
    uv = uv_grid_flat[uv_idx]

    # NOTE: MAX_RASTER_DEPTH lives in global scope
    # inds, valid = rasterize_result._i(uv_idx, MAX_RASTER_DEPTH) 
    inds, _ = rasterize_result.i(uv, MAX_RASTER_DEPTH) 

    sigmas = scene2d.w[inds]*mvnormal_pdf(uv, scene2d.mu[inds], scene2d.cov[inds])
    alphas = 1.0 - jnp.exp(-sigmas)

    _, key = jax.random.split(key)
    sampled = jax.random.bernoulli(key, p=alphas)
    colors = scene2d.c[inds]
    zs_ = jnp.where(sampled, zs[inds], jnp.inf)
    c = jnp.where(jnp.any(sampled), colors[jnp.argmin(zs_)], jnp.ones(3))

    return c

def _sample_image(key, tf, uv_grid_flat, scene, zs, rasterize_result, intr):
    (mus2d, covs2d), _ = project_gausians(scene.mu, scene.cov, tf, intr)
    scene2d = GaussianScene(mus2d, covs2d, scene.c, scene.weights)
    K = uv_grid_flat.shape[0]
    keys = jax.random.split(key, K)
    cs = jax.vmap(_sample_pixel, 
        (0, 0, None, None, None, None))(keys, 
        jnp.arange(K), uv_grid_flat, scene2d, zs, rasterize_result)

    return cs.reshape((intr.w, intr.h, 3)).transpose([1,0,2])


In [ ]:
tf = Transform(cam.t.copy(), cam.q.copy())
(mus2d, covs2d), valid = project_gausians(mus, covs, tf, intr)

# Rasterize
timer()
bbs = jax.block_until_ready(quantize_and_clip_bbs(bounding_boxes(mus2d, covs2d), intr.w, intr.h))
whs = bbs[:,1]-bbs[:,0]
areas = whs[:,0]*whs[:,1]
M = int(jnp.sum(areas))
zs = jax.block_until_ready(compute_depths(mus, tf))
rasterize_result = jax.block_until_ready(rasterize(bbs, zs, M, W, H))
timer("Rasterize")
# UVs
uv_grid_flat = compute_uv_grid(intr, flat=True, indexing="uv")

# Gaussian Scene
scene = GaussianScene(mus, covs, cols, 10*jnp.ones_like(mus[...,0]))

In [ ]:
key,_ = jax.random.split(key)
keys = jax.random.split(key, 100)


sampling_context = (tf, uv_grid_flat, scene, zs, rasterize_result, intr)
sample_many = jax.vmap(
    lambda key: _sample_image(key,*sampling_context))


im = _sample_image(key, *sampling_context)
ims = sample_many(keys)
figs = []
for i in np.arange(ims.shape[0], step=5):

    # ================================
    fig, axs = plt.subplots(1,2, figsize=(10,3))
    axs[0].imshow(ims[i]);
    axs[1].imshow(ims[:i].mean(0));
    # axs[0].axis("off")
    # axs[1].axis("off")
    fig.canvas.draw()

    buf = np.asarray(fig.canvas.buffer_rgba()) 
    img = buf[:, :, :3]
    figs.append(img)
    plt.close(fig)

In [ ]:
import imageio
from IPython.display import Image, display

# Needs to be of type uint8
imageio.mimsave("_animation.gif", figs, duration=0.1, loop=0)

display(Image(filename='_animation.gif'))

#### Logpdf

In [ ]:
#|export

# TODO: How can we enable easier debugging, add a debug:dict|None argument?
#   Always hand this down? 
def _compute_mixture_log_weights(sigmas, valid):
    # Compute transmittance
    sigmas = jnp.where(valid, sigmas, 0.0)
    log_transmittance = - jnp.cumulative_sum(sigmas, include_initial=True)[:-1]

    # Compute stopping weights
    alphas = 1.0 - jnp.exp(-sigmas)
    # TODO: log_alpha had -infs that were causing trouble.
    alphas = jnp.clip(alphas, 1e-9, 1.0)


    log_weights = log_transmittance + jnp.log(alphas)

    return jnp.where(valid, log_weights, -jnp.inf)

def _logpdf_cuv(c, uv, inds, valid, scene2d):
    # TODO: add Gaussian weights, should add to scene
    sigmas = mvnormal_pdf(uv, scene2d.mu[inds], scene2d.cov[inds])
    log_feature_probs = mvnormal_logpdf(c, scene2d.c[inds], .1)


    # Compute mixture weights and mixture score (inliner)
    # Note that we omit the background model here
    log_weights = _compute_mixture_log_weights(sigmas, valid)
    inlier_score = jax.nn.logsumexp(log_weights + log_feature_probs, b=valid)

    # "Compute" outlier score
    outlier_score = jnp.log(1.0)

    # Compute final inlier/outlier mixture
    score = jnp.logaddexp(
        jnp.log(0.5) + inlier_score, 
        jnp.log(0.5) + outlier_score)

    return score


def _score_per_pixel(uv_idx, im, uvs, scene2d, rasterize_result):

    uv = uvs[uv_idx]
    c = im[uv[1].astype(jnp.int32), uv[0].astype(jnp.int32)]

    # NOTE: MAX_RASTER_DEPTH lives in global scope
    # inds, valid = rasterize_result._i(uv_idx, MAX_RASTER_DEPTH) 
    inds, valid = rasterize_result.i(uv, MAX_RASTER_DEPTH) 
    
    score = _logpdf_cuv(c, uv, inds, valid, scene2d)

    return score

def _test_per_pixel(tf, uv_idx, im, uv_grid_flat, scene2d, rasterize_result, intr):
    (mus2d, covs2d), _ = project_gausians(scene.mu, scene.cov, tf, intr)
    scene2d = GaussianScene(mus2d, covs2d, scene.c, scene.weights)
    return _score_per_pixel(uv_idx, im, uv_grid_flat, scene2d, rasterize_result)

# TODO: Note this depends on a rasterization result, and 
#   doesn't compute it for the given tf. This is intentional because 
#   the rasterize_result defines only the region of influence, which 
#   is fixed for optimization.
def _splatting_scorer(tf, im, uv_grid_flat, scene, rasterize_result, intr):
    (mus2d, covs2d), _ = project_gausians(scene.mu, scene.cov, tf, intr)

    scene2d = GaussianScene(mus2d, covs2d, scene.c, scene.weights)
    scores = jax.vmap(_score_per_pixel, 
        (0, None, None, None, None))(jnp.arange(uv_grid_flat.shape[0]), 
            im, uv_grid_flat, scene2d, rasterize_result)

    return scores
            
splatting_scorer = jax.jit(_splatting_scorer)

##### Logpdf Test

**Get a scene and everything set up**

In [ ]:
# Sample a scene of N Gaussians (means, covariances, colors)
N = 10
key, key0, key1, key2 = jax.random.split(key, 4)
mus = jax.random.uniform(key0, (N,3), minval=-5, maxval=5)
covs = sample_covariance(key1, N, smin=jnp.array([0.1, 0.1, 0.05]), smax=jnp.array([0.5, 2., 1.]))
cols = jax.random.uniform(key2, (N,3), minval=0., maxval=1.)

# Camera
tf = Transform(cam.t.copy(), cam.q.copy())

# Gaussian Scene
scene = GaussianScene(mus, covs, cols, 10.*jnp.ones_like(mus[...,0]))
(mus2d, covs2d), valid = project_gausians(mus, covs, tf, intr)
scene2d = GaussianScene(mus2d, covs2d, cols, scene.weights)

# Memory Layout
bbs = quantize_and_clip_bbs(bounding_boxes(mus2d, covs2d), intr.w, intr.h)
whs = bbs[:,1]-bbs[:,0]
areas = whs[:,0]*whs[:,1]
M = int(areas.sum())
ms, inds, off = memory_layout(areas, M)

console.inspect(areas=areas, M=M, ms=ms, inds=inds, off=off)

# Rasterize
zs = compute_depths(mus, tf)
rasterize_result = rasterize(bbs, zs, M, W, H)
console.inspect(rasterize_result_counts=rasterize_result._counts)

# UVs
uv_grid_flat = compute_uv_grid(intr, flat=True, indexing="uv")

# Pseudo Render image
sampling_context = (tf, uv_grid_flat, scene, zs, rasterize_result, intr)
im = _sample_image(key,*sampling_context)

# ==========================
fig, axs = plt.subplots(1,3, figsize=(8,4))
axs[0].imshow(im)
axs[1].imshow(rasterize_result._counts2d.transpose(1,0))
axs[2].hist(rasterize_result._counts, bins=100);

**Run a per pixel test**

In [ ]:
uv_inds = jnp.where(rasterize_result >0)[0]
uv_idx = uv_inds[0]
context = (uv_idx, im, uv_grid_flat, scene2d, rasterize_result, intr)

console.print("[bold]Score:[/bold]", _test_per_pixel(tf, *context))
console.print("[bold]Gradient:[/bold]", jax.grad(
        lambda tf: _test_per_pixel(tf, *context))(tf))

# ===================================================
plt.figure(figsize=(5,2))
plt.imshow(rasterize_result._counts2d.transpose(1,0), extent=(0,intr.w, intr.h,0))
plt.scatter(uv_grid_flat[uv_idx,0], uv_grid_flat[uv_idx,1], c='r', s=20., marker='x');

**Full score test**

In [ ]:
# Argument Context
timer()
context = (im, uv_grid_flat, scene, rasterize_result, intr)
scs = jax.block_until_ready(splatting_scorer(tf, *context))
timer("Splatting Scorer", msg=scs.shape)
console.inspect(scs=scs)
console.print("[bold]Sum of scores:[/bold]", scs.sum())

# ============================================
fig, axs = plt.subplots(1,2,figsize=(10,5))

axs[0].set_title("Naive rendered image")
axs[0].set_aspect("equal")
axs[0].imshow(im, interpolation="none")

axs[1].set_title("Per pixel Scores")
axs[1].set_aspect("equal")
axs[1].imshow(scs.reshape(intr.resolution).T, interpolation="none", alpha=1.)

**Gradient test**

In [ ]:
score_many = jax.jit(jax.vmap(
    lambda tf: splatting_scorer(tf, *context).sum()))

In [ ]:
num_samples = 100
key, _ = jax.random.split(key)
ts = jax.random.uniform(key, (num_samples, 3), minval=-0.01, maxval=0.01)
qs = jax.random.uniform(key, (num_samples, 4), 
                        minval=(-.000)*jnp.ones(4), maxval=0.000*jnp.ones(4))
qs = qs.at[:,-1].set(1.)
qs = qs / jnp.linalg.norm(qs, axis=-1, keepdims=True)


tf0 = Transform(
    cam.t.copy() + jnp.array([0.0,0.5, 0.0]), 
    cam.q.copy()
)

tfs = tf0[None]._compose(Transform(ts, qs))


In [ ]:
scs = score_many(tfs)
order = jnp.argsort(scs)

scs = scs - jax.nn.logsumexp(scs)

# ============
plt.figure(figsize=(6,2))
plt.plot(jnp.exp(scs)[order], marker='o')

In [ ]:
ts = (tf0.inv()@tfs).t
qs = (tf0.inv()@tfs).q


plt.gca().set_aspect("equal")
plt.scatter(
    jnp.linalg.norm(ts[order], axis=-1), 
    scs[order], cmap="viridis", s=20.)


In [ ]:
ts = (tfs).t
qs = (tfs).q

console.print(jax.grad(
    lambda tf: splatting_scorer(tf, *context).sum())(tf0))

fig, axs = plt.subplots(1,2, figsize=(10,5))
axs[0].set_aspect("equal")
axs[0].scatter(ts[order,0], ts[order,1], c=scs[order], cmap="viridis", s=100.)

axs[1].set_aspect("equal")
axs[1].scatter(ts[order,1], ts[order,2], c=scs[order], cmap="viridis", s=100.)


In [ ]:
scs = splatting_scorer(tf0, *context)

In [ ]:

scs = splatting_scorer(tf0, *context)
console.print(scs.sum())
console.inspect(scs = scs)
console.print(jax.grad(
    lambda tf: splatting_scorer(tf, *context).sum())(tf0))

# ============================================
fig, axs = plt.subplots(1,2,figsize=(10,5))

axs[0].set_xlim(0, intr.w)
axs[0].set_ylim(0, intr.h)
axs[0].set_aspect("equal")
axs[0].imshow(im, interpolation="none")

axs[1].set_xlim(0, intr.w)
axs[1].set_ylim(0, intr.h)
axs[1].set_aspect("equal")
axs[1].imshow(scs.reshape(intr.resolution).T, interpolation="none", alpha=1.)

##### Debug

In [ ]:
uv_inds = jnp.where(rasterize_result == 1)[0]
err_idx = []
for uv_idx in uv_inds:
    context = (uv_idx, im, uv_grid_flat, scene2d, rasterize_result, intr)
    grad = jax.grad(lambda tf: _test_per_pixel(tf, *context))(tf)
    if jnp.all(jnp.isnan(grad.t)):
        err_idx.append(uv_idx)
        break

console.print(f"err_idx = {len(err_idx)}")

# # ===================================================
plt.imshow(rasterize_result._counts2d.transpose(1,0))
plt.scatter(uv_grid_flat[err_idx,0]-0.5, uv_grid_flat[err_idx,1]-0.5, c='r', s=10., marker='x')
# ===================================================

# Archive:

## Splatting Renderer: `fori_loop`

In [ ]:
@jax.jit
def f(n):
    return jax.lax.fori_loop(0, n, lambda i, carry: carry+i, 0)

ds = []
ns = jnp.arange(2_000, step=100)
# JIT pre-compile
jax.block_until_ready(f(ns[0]))
for n in ns:
    timer()
    result = jax.block_until_ready(f(n))
    d = timer("fori_loop", verbose=False, record=False)
    ds.append(d*1e3)


slope = (ds[-1]-ds[0])/(ns[-1]-ns[0])

# console.print(f"Counts max = {counts.max()} ~ {counts.max()*slope:.0f} ms")
# =============================
plt.figure(figsize=(5,1))
plt.plot(ns, ds)
plt.plot(ns, ns*slope, ":", color="C1")
plt.ylabel("Time [ms]")
plt.xlabel("N");

JAX `fori_loop` mechanics:
```Python
def fori_loop(lower, upper, body_fun, init_val):
  val = init_val
  for i in range(lower, upper):
    val = body_fun(i, val)
  return val
```

In [ ]:
#|export
# TODO: Make a RenderingContext class, for the `body_fun`
#   in the renderer loop to consume.

def _body_func(j, logp_ref):
    color, total_weight, log_tr, ref = logp_ref
    i, pixel_uvs, counts, hooks, order, gaussian_ids, mus2d, covs2d_inv, cols = ref

    uv = pixel_uvs[i]
    col = cols[gaussian_ids[order[hooks[i] + j]]]
    mu = mus2d[gaussian_ids[order[hooks[i] + j]]]
    cov_inv = covs2d_inv[gaussian_ids[order[hooks[i] + j]]]

    x = uv - mu
    d = jnp.exp(-0.5*(x.T@cov_inv@x))
    op = 1 - jnp.exp(-d)

    total_weight += jnp.exp(log_tr)*op
    color += jnp.exp(log_tr)*op*col
    log_tr -= d

    return (color, total_weight, log_tr, ref)

def _render_pixel(i, pixel_uvs, counts, hooks, order, gaussian_ids, mus2d, covs2d_inv, cols):

    bg_color = jnp.array([1.0, 1.0, 1.0])

    ref = (i, pixel_uvs, counts, hooks, order, gaussian_ids, mus2d, covs2d_inv, cols)

    # TODO: The `fori_loop` is the main bottleneck for performance.
    #   Can we upper bound counts[i], may be by only addressing
    #   the high density contributions?
    c, w, log_tr, _ = jax.lax.fori_loop(0, counts[i], _body_func,
        (jnp.array([0.0,0.0,0.0]), jnp.array([0.0]), jnp.array([0.]), ref))

    c += jnp.exp(log_tr)*bg_color

    w += jnp.exp(log_tr)

    return c, w, log_tr


render_splats = jax.jit(jax.vmap(_render_pixel, (0, None, None, None, None, None, None, None, None)))

In [ ]:
u, v = jnp.mgrid[:intr.w, :intr.h]
pixel_uvs = jnp.stack([u, v], axis=-1) + 0.5
pixel_uvs = pixel_uvs.reshape(-1,2).astype(jnp.float64)

In [ ]:
timer()
(mus2d, covs2d), valid = jax.block_until_ready(
    project_gausians(mus, covs, cam, intr))
timer("Project Gaussians")

# NOTE: We're quantizing the bounding boxes. That's the reason
#   we need to adjust the integer widths and heights by one.
bbs = bounding_boxes(mus2d, covs2d).astype(jnp.int32)
bbs = jnp.clip(bbs, 0, jnp.array([intr.w-1, intr.h-1]))
whs = (bbs[:,1]-bbs[:,0])+1
# TODO: This is not correct, Gaussians that land outside
# the image are invalid. However, if their bounding box
# is visible, they should still contribute.
# We need to add a mask or so.
areas = whs[:,0]*whs[:,1]
M = jnp.sum(areas).item()
ms, gs, off = memory_indices(areas, M)

timer()
zs = compute_depths(mus, cam)
counts, hooks, pxs, gs, order = jax.block_until_ready(
    rasterize(bbs, zs, M, intr.w, intr.h))
timer("Rasterize")

console.inspect(mus=mus, covs=covs, counts=counts, hooks=hooks, pxs=pxs, gs=gs, order=order)

In [ ]:
timer()
covs2d_inv = jax.block_until_ready(
    jnp.linalg.inv(covs2d))
timer("Inv Covs")

timer()
scorer_args = (jnp.arange(intr.w*intr.h), pixel_uvs, counts, hooks, order, gs, mus2d, covs2d_inv, cols)
im, ws, log_tr = jax.block_until_ready(
    render_splats(*scorer_args))
timer("Scorer", msg={"counts min/max": (counts.min(), counts.max())})

console.inspect(N=N, M=M, im=im, ws=ws, tr=jnp.exp(log_tr))

im = im[:,:].clip(0.,1.).reshape(intr.w,intr.h,3).transpose(1,0,2)

# ============================
fig, ax = plt.subplots(1,1, figsize=(20,20))
ax.imshow(im, interpolation="none")

In [ ]:
def convert_to_grayscale(im):
    return jnp.dot(im, jnp.array([0.2989, 0.5870, 0.1140]))


In [ ]:
gx = jnp.array([
    [-1, 0 , +1],
    [-2, 0 , +2],
    [-1, 0 , +1],
])

gy = gx.T

timer()
imx = jax.scipy.signal.convolve2d(convert_to_grayscale(im), gx, mode="same")
imy = jax.scipy.signal.convolve2d(convert_to_grayscale(im), gy, mode="same")
timer("Convolve (Sobel)")

ima = jnp.arctan2(imy, imx)
ima = (imx**2+imy**2)**0.5
timer("Arctan2")

console.inspect(ima=ima)

fig, axs = plt.subplots(4,1, figsize=(10,20))
axs[0].imshow(im, interpolation="none")
axs[1].imshow(imx, interpolation="none")
axs[2].imshow(imy, interpolation="none")
axs[3].imshow(ima, interpolation="none")
fig.tight_layout()